In [189]:
import numpy as np
import pandas as pd
import cvxpy as cp
import mosek
import matplotlib.pyplot as plt
import datetime as date
from datetime import datetime as dt
from dateutil.relativedelta import *
from scipy.stats import rankdata

In [206]:
################ all h functions evaluation

def h_quad(x,par):
    return((1+par)*x-par*x**2)

def h_single_power(x,par):
    return(1-(1-x)**par)


In [207]:
################ all phi functions constraints

def modified_chi2(p_i,q_i,par,phi_cons):
    phi_cons = phi_cons + 1/p_i*(q_i-p_i)**2
    return(phi_cons)

def kullback_leibler(p_i,q_i,par,phi_cons):
    phi_cons = phi_cons -cp.entr(q_i) - q_i*np.log(p_i)
    return(phi_cons)

In [229]:
############### all h functions constraints

def h_quadratic(z1,z2,par,constraints):
    v = (1+par)*cp.sum(z2)-par*cp.sum(z2)**2
    constraints.append(cp.sum(z1)-v <= 0)
    return(constraints)

def h_sing_power(z1,z2,par,constraints):
    v = 1-cp.power((1-cp.sum(z2)),par)
    constraints.append(cp.sum(z1)-v <= 0)
    return(constraints)

In [219]:
############## all phi conjugates

def modified_chi2_conj(gamma, s,t, constraints):
    N = s.shape[0]
    w = cp.Variable(N, nonneg = True)
    for i in range(N):
        constraints.append(cp.norm(cp.vstack([w[i],t[i]/2]))<=(t[i]+2*gamma)/2)
    constraints.append(s/2+gamma*(np.zeros(N)+1)<= w)
    return(constraints)

def kullback_leibler_conj(gamma,s,t,constraints):
    N = s.shape[0]
    w = cp.Variable(N)
    constraints.append(w - gamma*(np.zeros(N)+1) <= t)
    for i in range(N):
        constraints.append(cp.kl_div(gamma,w[i])+gamma+s[i]-w[i]<= 0)
    return(constraints)

In [220]:
############## all h conjugates

def h_quad_conj(lbda,v,z,par,constraints):
    M = lbda.shape[0]
    eta = cp.Variable(M, nonneg= True)
    for j in range(M):
        constraints.append(cp.norm(cp.vstack([eta[j],(z[j]-lbda[j])/2]))<=(z[j]+lbda[j])/2)
        constraints.append(1/(2*np.sqrt(m))*(-v[j]+lbda[j]+par*lbda[j])<= eta[j])
    return(constraints)

def h_sing_power_conj(lbda,v,z,par,constraints):
    M = lbda.shape[0]
    xi_2 = cp.Variable(M, nonneg = True)
    xi_3 = cp.Variable(M, nonneg = True)
    xi_4 = cp.Variable(M, nonneg = True)
    constraints.extend((xi_3 <= xi_2, xi_3 <= -v))
    constraints.append(lbda-xi_3+(par**(-1/(par-1))-par**(-par/(par-1)))*xi_4 <= z)
    exponent = np.array([(par-1)/par,1-(par-1)/par])
    for j in range(M):
        constraints.append(xi_2[j]-cp.geo_mean(cp.vstack([xi_4[j],lbda[j]]),exponent)<= 0)
    return(constraints)
        

In [221]:
############### all utility functions

def lin_utility(R,a,r_f,par):
    return(R@a + (1-cp.sum(a))*r_f)

def lin_utility_eva(R,a,r_f,par):
    return(R.dot(a)+(1-sum(a))*r_f)
    

In [222]:
############## all set operations

def ranktoset (A):
    A = list(A)
    sets = [[A[0]]]
    for i in range(1,len(A)):
        new = A[0:i+1]
        sets.append(new)
    return(sets)

def makesetflex (A, B):    # we assume A is non-empty
    N = len(A)
    B = list(B)
    M = len(B)
    added = []
    for i in range(M):
        new = B[0:i+1]
        for k in range(N):
            if len(A[k])==len(new) and len(np.intersect1d(A[k],new))==len(new):
                break
            if k == N-1:
                A.append(new)
                added.append(new)
    return(A,added)

In [223]:
############### robust_counterpart and robustness check


def robust_counterpart(sets,p,R,r,par,r_f,c,phi_conj, h_conj,utility):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    v = cp.Variable(M)
    lbda = cp.Variable(M, nonneg = True)
    a = cp.Variable(I)
    alpha = cp.Variable(1)
    beta = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N)
    z = cp.Variable(M)
    s = cp.Variable(N)
    constraints = [cp.abs(a)<= 10]
    for i in range(N):
        lbdasum = 0
        vsum = 0
        for j in range(M):
            if i in sets[j]:
                lbdasum = lbdasum + lbda[j]
                vsum = vsum + v[j]
        constraints.append(-utility(R,a,r_f,par)[i] - lbdasum - beta <= 0)
        constraints.append(s[i] == -alpha + vsum)
    constraints = phi_conj(gamma,s,t,constraints)
    constraints = h_conj(lbda,v,z,par,constraints)
    constraints.append(alpha + beta + gamma * r  + cp.sum(z) + p@t <= c)
    obj = cp.Maximize((utility(R,a,r_f,par)).T @ p)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(a.value, prob.value)
    
def robustcheck(a,R,r,p,par,r_f,h_func, phi_func,utility, utility_eva):
    N = len(p)
    x = -utility_eva(R,a,r_f,par)
    rank = np.argsort((-x))
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1, cp.sum(q_b)==1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        constraints = h_func(z1,z2,par,constraints)
        phi_cons = phi_func(p[i],q[i],par,phi_cons)
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(-q_b.T @ utility(R,a,r_f,par))
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value,q_b.value)



In [224]:
################### final algorithm

def squeeze_algorithm(R,r,c,p,par,r_f,h_eva, phi_func,h_func,phi_conj,h_conj,utility):
    N = len(p)
    I = len(R[0])
    a = cp.Variable(I)
    constraints = [cp.abs(a)<=10]
    h = np.zeros(N)
    iterations = 0
    steps = 0
    for i in range(N-1):
        h[i] = h_eva(sum(p[i:N]),par)-h_eva(sum(p[i+1:N]),par)
    h[N-1]=h_eva(p[N-1],par)
    constraints.append(-h.T@(R @ a)-(1-cp.sum(a))*r_f<= c)
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    w = a.value
    upperobj = prob.value
    nonstop = True
    while nonstop:
        [rbvalue,h] = robustcheck(w,R,r,p,par,r_f,h_func,phi_func,utility,utility_eva)
        print('rbvalue',rbvalue)
        if rbvalue <= c+1e-3:
            oldrank = np.argsort(R.dot(w))
            sets = ranktoset(oldrank)
            while nonstop:
                [w,lowerobj] = robust_counterpart(sets,p,R,r,par,r_f,c,phi_conj,h_conj,utility)
                newrank = np.argsort(R.dot(w))
                if np.array_equal(newrank,oldrank):
                    break
                [sets,added] = makesetflex(sets,newrank)
                oldrank = newrank
                steps = steps + 1
            return(w,'upperbound', upperobj, 'lowerbound', lowerobj, 'cut-iterations', iterations,' robust iterations' ,steps)
        constraints.append(-h.T@(R @ a)-(1-cp.sum(a))*r_f<= c)
        iterations = iterations + 1
        prob = cp.Problem(obj,constraints)
        prob.solve(solver=cp.MOSEK)
        w = a.value
        upperobj = prob.value
        print('upperbound' , upperobj, 'cut-iterations', iterations)

In [225]:
def non_robustcheck_chi2(a,R,p,m,r_f):
    N = len(p)
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    q_b = cp.Variable(N, nonneg = True)
    constraints = [cp.sum(q_b)==1]
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = p[rank[0:i+1]]
        v = (1+m)*cp.sum(z2)-m*cp.sum(z2)**2
        constraints.append(cp.sum(z1)-v <= 0)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value - (1-np.sum(a))*r_f,q_b.value)


def non_robust_cut_chi2(R,c,p,m,r_f):
    N = len(p)
    I = len(R[0])
    a = cp.Variable(I)
    constraints = [cp.abs(a)<=10]
    h = np.zeros(N)
    iterations = 0
    for i in range(N-1):
        h[i] = h_quad(sum(p[i:N]),m)-h_quad(sum(p[i+1:N]),m)
    h[N-1]=h_quad(p[N-1],m)
    constraints.append(-h.T@(R @ a)-(1-cp.sum(a))*r_f<= c)
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    w = a.value
    nonstop = True
    while nonstop:
        [rbvalue,h] = non_robustcheck_chi2(w,R,p,m,r_f)
        print('rbvalue',rbvalue)
        if rbvalue <= c +1e-3:
            return(w,'upperbound', upperobj, 'cut-iterations', iterations)
        constraints.append(-h.T@(R @ a)-(1-cp.sum(a))*r_f<= c)
        iterations = iterations + 1
        prob = cp.Problem(obj,constraints)
        prob.solve(solver=cp.MOSEK)
        w = a.value
        upperobj = prob.value
        print('upperbound' , upperobj, 'cut-iterations', iterations)

In [123]:
df_returns6 = pd.read_csv('6_Portfolios_2x3.csv', skiprows = 15)

In [124]:
df_returns = df_returns6[0:1144].copy()
df_returns['Date'] = pd.to_datetime(df_returns['Date'], format = '%Y%m')
for i in range(1, len(df_returns.columns)):
    df_returns[df_returns.columns[i]] = pd.to_numeric(df_returns[df_returns.columns[i]])
df_returns

,Date,SMALL LoBM,ME1 BM2,SMALL HiBM,BIG LoBM,ME2 BM2,BIG HiBM
0,1926-07-01,1.0874,0.9349,-0.0695,5.7168,1.9620,1.4222
1,1926-08-01,0.7030,1.2300,5.3842,2.7154,2.6930,6.3154
2,1926-09-01,-2.9117,-0.1303,-0.4374,1.4287,0.0704,-0.7967
3,1926-10-01,-3.8196,-4.5860,-2.0112,-3.5898,-2.3398,-4.0970
4,1926-11-01,3.1806,3.7233,2.0944,3.1292,2.8952,3.4614
...,...,...,...,...,...,...,...
1139,2021-06-01,5.6058,0.4400,-1.0979,4.8188,-1.2594,-4.0036
1140,2021-07-01,-5.5593,-1.8623,-3.6521,3.1048,-0.0099,-2.3000
1141,2021-08-01,2.3903,1.5124,2.6680,3.5667,1.4122,3.0371
1142,2021-09-01,-4.3421,-3.4661,0.6445,-5.4525,-3.8570,-0.2526


In [200]:
startdate = dt(2018,1,3)
X = df_returns[df_returns.Date >= startdate][df_returns.columns[1:7]]
X = X.reset_index(drop = True)
R = X.to_numpy()

In [230]:
N=R.shape[0]
p = np.zeros(N)+1/N
I = R.shape[1]
par = 4
r = 0.05   # this is the parameter of the h function min(1, p/(1-m))
r_f = 0.07
c = 10
h_eva = h_single_power
phi_func = kullback_leibler
h_func = h_sing_power
phi_conj = kullback_leibler_conj
h_conj = h_sing_power_conj
utility= lin_utility
utility_eva = lin_utility_eva

In [231]:
squeeze_algorithm(R,r,c,p,par,r_f,h_eva, phi_func,h_func,phi_conj,h_conj,utility)

rbvalue 546.0952181430248
upperbound 25.89585829020141 cut-iterations 1
rbvalue 92.04780585859045
upperbound 16.357109934633794 cut-iterations 2
rbvalue 31.09264116657512
upperbound 14.896591528928452 cut-iterations 3
rbvalue 23.979764302609524
upperbound 9.74991391032086 cut-iterations 4
rbvalue 26.265561878924476
upperbound 8.763181736983615 cut-iterations 5
rbvalue 15.842484546790132
upperbound 8.42298047884612 cut-iterations 6
rbvalue 14.200753953087252
upperbound 8.343212316499088 cut-iterations 7
rbvalue 12.833480434574534
upperbound 8.09332237241574 cut-iterations 8
rbvalue 11.970229195404308
upperbound 7.887565149190319 cut-iterations 9
rbvalue 11.317765290601365
upperbound 7.681686301606384 cut-iterations 10
rbvalue 11.227817933680473
upperbound 7.611319446577719 cut-iterations 11
rbvalue 11.832500862330964
upperbound 7.592400676075687 cut-iterations 12
rbvalue 10.95866588660673
upperbound 7.466133725843085 cut-iterations 13
rbvalue 10.687734125921738
upperbound 7.387229656640

(array([10., 10., 10., 10., 10., 10.]),
 'upperbound',
 7.201762199312931,
 'lowerbound',
 72.55537777777778,
 'cut-iterations',
 44,
 ' robust iterations',
 1)

In [73]:
results_nonrb = non_robust_cut_chi2(R,c,p,m,r_f)
print(results_nonrb)
robustcheck_chi2(results_nonrb[0],R,r,p,m,r_f)[0]

rbvalue 85.67052759192939
upperbound 20.72240497618754 cut-iterations 1
rbvalue 14.744315380125778
upperbound 17.649127488790377 cut-iterations 2
rbvalue 13.466294381999045
upperbound 17.409629150142457 cut-iterations 3
rbvalue 11.038432258740148
upperbound 17.15855968539835 cut-iterations 4
rbvalue 10.483893630682717
upperbound 17.012099884975086 cut-iterations 5
rbvalue 10.318094939506024
upperbound 16.89003492916256 cut-iterations 6
rbvalue 10.164542693501566
upperbound 16.8680838793589 cut-iterations 7
rbvalue 10.07789992592972
upperbound 16.804398042247232 cut-iterations 8
rbvalue 10.042023083557499
upperbound 16.803788673060602 cut-iterations 9
rbvalue 10.019832081166008
upperbound 16.800732344467765 cut-iterations 10
rbvalue 10.010729987173443
upperbound 16.800671728312363 cut-iterations 11
rbvalue 10.004845562903434
upperbound 16.79965053155821 cut-iterations 12
rbvalue 10.024616424037786
upperbound 16.799392982449366 cut-iterations 13
rbvalue 10.0065801535453
upperbound 16.799

44.33872406427757

In [28]:
cons = []
x= [1,2,3]
for i in range(3):
    cons.append(x[i])

[Inequality(Variable((1,))), Inequality(Variable((1,)))]